# 02 — Retrieve-and-Rerank

```
╔═══════════════════════════════════════════════════════════════════╗
║               2. RETRIEVE-AND-RERANK                              ║
║                                                                   ║
║  Documents ──► Chunks ──► Embedding Model ──► Vector DB          ║
║                                  ▲          top-20│              ║
║  Query ─────────────────────────►│               ▼              ║
║                                          Retrieved Context       ║
║                                          (Top-k)                 ║
║                                               │                  ║
║                                               ▼                  ║
║                                        Reranker Model            ║
║                                               │                  ║
║                                               ▼                  ║
║  Response ◄── Generative Model ◄── Prompt ◄── Re-ranked Context  ║
╚═══════════════════════════════════════════════════════════════════╝
```

## The problem with Naive RAG retrieval

The bi-encoder (sentence-transformer) embeds documents **independently** of the query. It's fast (O(n) lookups), but it misses fine-grained relevance: it captures whether a chunk is *about the same topic* but not whether it *directly answers the question*.

A **cross-encoder reranker** solves this by scoring each `(query, passage)` pair jointly — it sees both at the same time, so it can spot exact answer phrases and nuanced relevance. The tradeoff: it's too slow to run against the entire corpus, so we use a 2-stage pipeline:

1. **Stage 1 — Retrieve**: Fast bi-encoder gets the top-20 candidates
2. **Stage 2 — Rerank**: Slow but precise cross-encoder scores each candidate against the query, picks top-5

## What you'll learn
- The difference between bi-encoders and cross-encoders
- How to build a 2-stage retrieve-then-rerank pipeline
- Visualise the ranking change (before vs after)
- When reranking is and isn't worth the extra latency

In [ ]:
import sys; sys.path.insert(0, '..')
import ragkit.config as cfg

cfg.BACKEND = "claude"   # "claude" | "local"
print(f"Backend: {cfg.BACKEND}  |  Device: {cfg.DEVICE}")

## Build the index (same as Notebook 01)

In [ ]:
from ragkit.data import build_chunked_corpus
from ragkit.vectorstore import build_collection

texts, metadatas = build_chunked_corpus(chunk_size=200, overlap=40)
collection = build_collection("helios_rerank", texts, metadatas, persist_dir="../.chroma")
print(f"Index ready: {collection.count()} chunks")

## Understand the two model types

### Bi-encoder (fast, independent encoding)

```
Query  ──► [Encoder] ──► q_vec ─┐
                                  ├── dot product → similarity score
Passage ──► [Encoder] ──► p_vec ─┘
```

### Cross-encoder (slow, joint encoding)

```
  [Query + Passage combined]
          │
     [Encoder]
          │
   relevance score (0–1)
```

The cross-encoder is 10–100× slower per query (can't precompute document representations), but 20–40% more accurate.

In [ ]:
import time
from ragkit.embeddings import embed
from ragkit.rerank import rerank

# Time the bi-encoder (embedding a batch)
t0 = time.time()
_ = embed(texts[:20])
t_bienc = time.time() - t0

# Time the cross-encoder (scoring 20 pairs)
from ragkit.rerank import _get_reranker
reranker = _get_reranker()
query = "What is the fix for Joint 4 temperature problem?"
pairs = [(query, t) for t in texts[:20]]
t0 = time.time()
_ = reranker.predict(pairs)
t_cross = time.time() - t0

print(f"Bi-encoder (20 passages):  {t_bienc*1000:.0f} ms")
print(f"Cross-encoder (20 pairs):  {t_cross*1000:.0f} ms")
print(f"Speed ratio: {t_cross/t_bienc:.1f}× slower")
print("\n→ This is why we only run the cross-encoder on the top-N bi-encoder candidates.")

## Stage 1 — Retrieve top-20 candidates

In [ ]:
from ragkit.vectorstore import query_collection
from ragkit.pretty import show_hits

query = "What is the fix for the Joint 4 temperature problem?"

candidates = query_collection(collection, query, k=20)
print(f"Bi-encoder retrieved {len(candidates)} candidates")
show_hits(candidates[:8], title="Top-8 bi-encoder candidates (before rerank)")

## Stage 2 — Rerank with the cross-encoder

In [ ]:
from ragkit.rerank import rerank

reranked = rerank(query, candidates, top_k=5)
show_hits(reranked, title="Top-5 after cross-encoder reranking")

## Before vs After comparison

In [ ]:
from ragkit.pretty import compare_rankings

compare_rankings(
    before=candidates[:5],
    after=reranked,
    title_before="Bi-encoder top-5",
    title_after="Cross-encoder top-5",
)

In [ ]:
# Visualise the rank changes as a Sankey-style diagram
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

before_ids = [h.metadata['source'] + f"_c{h.metadata['chunk']}" for h in candidates[:10]]
after_ids  = [h.metadata['source'] + f"_c{h.metadata['chunk']}" for h in reranked]

fig, ax = plt.subplots(figsize=(9, 5))
ax.set_xlim(0, 10); ax.set_ylim(-0.5, len(before_ids) - 0.5)
ax.axis('off')
ax.set_title('Rank changes: bi-encoder → cross-encoder (top 10)', fontsize=11)

colors = ['#27ae60', '#2ecc71', '#f39c12', '#e74c3c', '#8e44ad',
          '#2980b9', '#16a085', '#d35400', '#7f8c8d', '#34495e']

for i, doc_id in enumerate(before_ids):
    short = doc_id.split('_c')[0].replace('spec_', '').replace('incident_', 'inc_')[:20]
    ax.text(0.1, len(before_ids)-1-i, f"{i+1}. {short}", va='center', fontsize=8, color=colors[i])

for j, doc_id in enumerate(after_ids):
    short = doc_id.split('_c')[0].replace('spec_', '').replace('incident_', 'inc_')[:20]
    ax.text(9.9, len(before_ids)-1-j, f"{j+1}. {short}", va='center', fontsize=8,
            ha='right', color=colors[before_ids.index(doc_id)] if doc_id in before_ids else 'black')

for j, doc_id in enumerate(after_ids):
    if doc_id in before_ids:
        i = before_ids.index(doc_id)
        y_from = len(before_ids) - 1 - i
        y_to   = len(before_ids) - 1 - j
        ax.annotate('', xy=(9.5, y_to), xytext=(0.5, y_from),
                    arrowprops=dict(arrowstyle='->', color=colors[i], lw=1.5, alpha=0.7))

ax.text(5, len(before_ids)-0.2, 'Reranking', ha='center', fontsize=9,
        bbox=dict(boxstyle='round', fc='#ecf0f1', ec='#bdc3c7'))
plt.tight_layout(); plt.show()

## Generate the answer

In [ ]:
from ragkit.llm import generate
from ragkit.pretty import show_answer

SYSTEM = """You are a helpful technical support assistant for Helios Robotics.
Answer questions using ONLY the provided context. Be concise and precise.
Mention relevant part numbers, firmware versions, or incident IDs."""

def retrieve_rerank_generate(question: str, collection, retrieve_k=20, final_k=5) -> str:
    # Stage 1: bi-encoder retrieval
    candidates = query_collection(collection, question, k=retrieve_k)
    
    # Stage 2: cross-encoder rerank
    top = rerank(question, candidates, top_k=final_k)
    
    # Stage 3: generate
    context = "\n\n".join(f"[{h.metadata['source']}]\n{h.text}" for h in top)
    prompt = f"Context:\n{context}\n\n---\nQuestion: {question}"
    return generate(prompt, system=SYSTEM)

answer = retrieve_rerank_generate(query, collection)
show_answer(answer)

## Side-by-side: Naive RAG vs Retrieve-and-Rerank

Run the same query through both pipelines and compare.

In [ ]:
from ragkit.pretty import show_answer

# Pick a query where reranking makes a difference:
# This query uses different vocabulary than the stored chunks
tricky_query = "How do I resolve the arm's heat-related torque alarm?"

# Naive: top-5 by bi-encoder only
naive_hits = query_collection(collection, tricky_query, k=5)
naive_context = "\n\n".join(f"[{h.metadata['source']}]\n{h.text}" for h in naive_hits)
naive_answer = generate(f"Context:\n{naive_context}\n\n---\nQuestion: {tricky_query}", system=SYSTEM)

# Retrieve-and-Rerank: top-20 bi-encoder → top-5 cross-encoder
rr_answer = retrieve_rerank_generate(tricky_query, collection)

print("=" * 60)
print("NAIVE RAG:")
print(naive_answer)
print("=" * 60)
print("RETRIEVE-AND-RERANK:")
print(rr_answer)

## When is reranking worth it?

Run a quick latency comparison:

In [ ]:
import time

q = "What is the gripper part number?"

t0 = time.time()
query_collection(collection, q, k=5)
t_naive = time.time() - t0

t0 = time.time()
cands = query_collection(collection, q, k=20)
rerank(q, cands, top_k=5)
t_rr = time.time() - t0

print(f"Naive retrieval:          {t_naive*1000:.0f} ms")
print(f"Retrieve-and-rerank:      {t_rr*1000:.0f} ms  ({t_rr/t_naive:.1f}× slower)")
print("\nConclusion: reranking adds ~100-300ms latency. Worth it when:")
print("  ✓ Query vocabulary differs from document vocabulary")
print("  ✓ Many semantically similar chunks (fine-grained ranking matters)")
print("  ✓ High-stakes answers where accuracy > speed")
print("  ✗ Not worth it for simple factual lookups with clear vocabulary match")

## Exercise — When the cross-encoder rescues the right answer (Spanish grammar)

A fast **bi-encoder** scores the query and each document *separately*, so it can be fooled by surface wording. A slower **cross-encoder** reads the query and document *together* and judges true relevance. This exercise shows a case where stage-1 retrieval ranks the wrong passage first, and reranking fixes it.

The question: **"When do I use *estar* for location?"** The correct answer is the rule that ties *estar* to *location*.

1. **Predict first**: Of the four passages below, which one actually answers the question? Which one might a shallow bi-encoder rank first just because it repeats the word *estar* a lot?
2. **Implement**: Complete `cross_score` so it rewards a passage for containing the concepts the query truly needs.
3. **Compare**: Confirm that the stage-1 (bi-encoder) top-1 is **wrong**, but the reranked top-1 is **right**.
4. **Reflect**: One sentence — why not just run the cross-encoder on the whole corpus and skip stage 1?

In [ ]:
candidates = {
    "A": "Use 'ser' for permanent traits, identity, and origin.",
    "B": "Use 'estar' for temporary states and physical location.",
    "C": "'Estar' is irregular: estoy, estas, esta, estamos.",
    "D": "Spanish has two verbs that both translate as 'to be'.",
}

# Stage-1 bi-encoder similarity (given). Note C and A score high on surface wording.
bi_score = {"A": 0.62, "B": 0.55, "C": 0.60, "D": 0.50}
stage1 = sorted(candidates, key=lambda c: bi_score[c], reverse=True)
print("Stage-1 (bi-encoder) order:", stage1, "→ top-1:", stage1[0])

query = "When do I use estar for location?"

# ── Task 2: a toy cross-encoder ───────────────────────────────────────────────
# Reward a passage for containing BOTH concepts the query needs.
def cross_score(q: str, doc: str) -> int:
    needed = ["estar", "location"]              # ← the concepts that truly matter
    return sum(1 for w in needed if w in doc.lower())

reranked = sorted(candidates, key=lambda c: cross_score(query, candidates[c]), reverse=True)
print("Reranked order:           ", reranked, "→ top-1:", reranked[0])
print("Cross scores:", {c: cross_score(query, candidates[c]) for c in candidates})

# ── Task 4: why keep stage 1 at all? (comment) ────────────────────────────────
#   Your answer:

# ── Self-check ────────────────────────────────────────────────────────────────
assert stage1[0] != "B", "bi-encoder should be fooled by a surface-similar passage"
assert reranked[0] == "B", "the cross-encoder should rescue the passage about location"
print("\n✅ Exercise checks passed!")

## Tradeoffs

| Aspect | Naive RAG | Retrieve-and-Rerank |
|---|---|---|
| **Retrieval quality** | ★★★☆☆ | ★★★★★ |
| **Latency** | ★★★★★ Fast | ★★★☆☆ +100-300ms |
| **Setup complexity** | ★☆☆☆☆ | ★★☆☆☆ |
| **Multi-hop** | ★☆☆☆☆ | ★☆☆☆☆ Still poor |
| **Exact string match** | ★★☆☆☆ | ★★☆☆☆ Still limited |
| **When to use** | Quick prototypes, low-latency apps | Production systems, high accuracy requirements |

## Exercises

1. **Vary `retrieve_k`**: Try `retrieve_k=5`, `10`, `30` with `final_k=5`. Does more initial candidates help?
2. **Different reranker**: Try `cross-encoder/ms-marco-MiniLM-L-12-v2` (larger, slower). Does it improve?
3. **Find a query where reranking hurts**: Can you construct a query where the bi-encoder top-1 was correct and reranking displaced it?
4. **Score distribution**: Plot the cross-encoder score distribution. Are the scores well-separated, or clustered?

**Next:** [03_multimodal_rag.ipynb](03_multimodal_rag.ipynb) — extend RAG to handle images.